# Task 2: Text Chunking, Embedding, and Vector Store Indexing
## Converting Complaint Narratives for Semantic Search

This notebook implements text chunking, embedding generation, and vector store creation for the RAG complaint analysis system.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from typing import List, Dict

# Import the Task 2 implementation
import sys
sys.path.append('../src')
from task2_embedding_pipeline import ComplaintVectorStore, TextChunker

## 1. Load Preprocessed Data

In [ ]:
# Load cleaned data from Task 1
data_path = Path('../data/processed/filtered_complaints.csv')
df = pd.read_csv(data_path)

print(f"Loaded dataset shape: {df.shape}")
print(f"Product distribution:")
print(df['product_category'].value_counts())

## 2. Stratified Sampling Strategy

**Sampling Strategy**: We use stratified sampling to maintain proportional representation across all product categories. This ensures our vector store reflects the true distribution of complaint types while reducing computational requirements.

**Benefits**:
- Maintains representative sample across products
- Reduces training time while preserving data diversity
- Enables fair evaluation across all product categories

In [ ]:
# Initialize vector store
vector_store = ComplaintVectorStore()

# Create stratified sample
sample_df = vector_store.create_stratified_sample(df, sample_size=12000)

# Visualize sampling results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Original distribution
df['product_category'].value_counts().plot(kind='bar', ax=ax1, title='Original Distribution')
ax1.set_xlabel('Product Category')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Sample distribution
sample_df['product_category'].value_counts().plot(kind='bar', ax=ax2, title='Sample Distribution')
ax2.set_xlabel('Product Category')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Calculate proportions
original_props = df['product_category'].value_counts(normalize=True)
sample_props = sample_df['product_category'].value_counts(normalize=True)

comparison_df = pd.DataFrame({
    'Original (%)': original_props * 100,
    'Sample (%)': sample_props * 100
})
comparison_df['Difference (%)'] = abs(comparison_df['Original (%)'] - comparison_df['Sample (%)'])

print("\nProportional Representation Comparison:")
print(comparison_df.round(2))

## 3. Text Chunking Strategy

**Chunking Approach**: We use a recursive character-based text splitter with the following parameters:
- **Chunk Size**: 500 characters - balances context preservation with embedding efficiency
- **Overlap**: 50 characters - ensures continuity across chunk boundaries
- **Word Boundary Splitting**: Prevents breaking words mid-sentence

**Justification**:
- 500 chars provides sufficient context for semantic understanding
- 50-char overlap maintains narrative flow across chunks
- Word boundary preservation improves embedding quality

In [ ]:
# Analyze narrative lengths before chunking
narrative_col = 'Consumer complaint narrative'
sample_df['narrative_length'] = sample_df[narrative_col].str.len()

print("Narrative length statistics (characters):")
print(sample_df['narrative_length'].describe())

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(sample_df['narrative_length'], bins=50, alpha=0.7, edgecolor='black')
plt.axvline(x=500, color='red', linestyle='--', label='Chunk Size (500)')
plt.xlabel('Narrative Length (characters)')
plt.ylabel('Frequency')
plt.title('Distribution of Narrative Lengths')
plt.legend()

plt.subplot(1, 2, 2)
sample_df.boxplot(column='narrative_length', by='product_category', ax=plt.gca())
plt.axhline(y=500, color='red', linestyle='--', label='Chunk Size')
plt.title('Narrative Length by Product Category')
plt.suptitle('')  # Remove default title
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

# Test chunking strategy
chunker = TextChunker(chunk_size=500, chunk_overlap=50)
sample_text = sample_df[narrative_col].iloc[0]
sample_chunks = chunker.split_text(sample_text)

print(f"\nSample narrative length: {len(sample_text)} characters")
print(f"Number of chunks created: {len(sample_chunks)}")
print(f"\nFirst chunk: {sample_chunks[0]}")
if len(sample_chunks) > 1:
    print(f"\nSecond chunk: {sample_chunks[1]}")

## 4. Embedding Model Selection

**Model Choice**: `sentence-transformers/all-MiniLM-L6-v2`

**Justification**:
- **Efficiency**: 384-dimensional embeddings balance quality with computational efficiency
- **Performance**: Strong semantic understanding for financial/business text
- **Size**: Compact model (~80MB) suitable for production deployment
- **Training**: Pre-trained on diverse text corpus including customer service content
- **Speed**: Fast inference suitable for real-time applications

In [ ]:
# Perform chunking on sample dataset
chunks = vector_store.chunk_complaints(sample_df)

# Analyze chunking results
chunk_lengths = [len(chunk['chunk_text']) for chunk in chunks]
chunks_per_complaint = {}

for chunk in chunks:
    complaint_id = chunk['complaint_id']
    if complaint_id not in chunks_per_complaint:
        chunks_per_complaint[complaint_id] = 0
    chunks_per_complaint[complaint_id] += 1

chunks_per_complaint_values = list(chunks_per_complaint.values())

# Visualize chunking statistics
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Chunk length distribution
ax1.hist(chunk_lengths, bins=30, alpha=0.7, edgecolor='black')
ax1.axvline(x=500, color='red', linestyle='--', label='Target Size')
ax1.set_xlabel('Chunk Length (characters)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Chunk Lengths')
ax1.legend()

# Chunks per complaint
ax2.hist(chunks_per_complaint_values, bins=20, alpha=0.7, edgecolor='black')
ax2.set_xlabel('Chunks per Complaint')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Chunks per Complaint')

# Chunks by product category
chunk_df = pd.DataFrame(chunks)
chunk_counts_by_product = chunk_df['product_category'].value_counts()
chunk_counts_by_product.plot(kind='bar', ax=ax3)
ax3.set_xlabel('Product Category')
ax3.set_ylabel('Number of Chunks')
ax3.set_title('Chunks by Product Category')
ax3.tick_params(axis='x', rotation=45)

# Summary statistics
stats_text = f"""Chunking Statistics:
Total Chunks: {len(chunks):,}
Avg Chunk Length: {np.mean(chunk_lengths):.0f} chars
Avg Chunks/Complaint: {np.mean(chunks_per_complaint_values):.1f}
Min Chunk Length: {np.min(chunk_lengths)}
Max Chunk Length: {np.max(chunk_lengths)}"""

ax4.text(0.1, 0.5, stats_text, transform=ax4.transAxes, fontsize=12, 
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightgray'))
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')
ax4.set_title('Summary Statistics')

plt.tight_layout()
plt.show()

print(f"Generated {len(chunks)} chunks from {len(sample_df)} complaints")
print(f"Average chunks per complaint: {len(chunks)/len(sample_df):.1f}")

## 5. Generate Embeddings and Create Vector Stores

In [ ]:
# Generate embeddings
embeddings = vector_store.generate_embeddings(chunks)

print(f"Embedding model: {vector_store.model_name}")
print(f"Embedding dimensions: {vector_store.embedding_dim}")
print(f"Generated embeddings shape: {embeddings.shape}")

# Create vector stores
vector_store.create_chromadb_store(chunks, embeddings)
vector_store.create_faiss_store(embeddings, chunks)

# Save to disk
vector_store_dir = Path('../vector_store')
vector_store.save_vector_stores(vector_store_dir)

## 6. Test Search Functionality

In [ ]:
# Test semantic search with sample queries
test_queries = [
    "credit card billing error",
    "loan application rejected", 
    "money transfer delayed",
    "unauthorized charges",
    "account closure issues"
]

print("Testing semantic search functionality:")
print("=" * 50)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 30)
    
    results = vector_store.search_similar(query, k=3, use_faiss=True)
    
    for i, result in enumerate(results, 1):
        print(f"{i}. [{result['product_category']}] Similarity: {result['similarity_score']:.3f}")
        print(f"   Issue: {result['issue']}")
        print(f"   Text: {result['chunk_text'][:150]}...")
        print()

## 7. Vector Store Comparison: ChromaDB vs FAISS

**ChromaDB**:
- Built-in metadata filtering
- Easy persistence and loading
- Good for development and prototyping

**FAISS**:
- High-performance similarity search
- Optimized for production workloads
- Better scalability for large datasets

In [ ]:
# Compare search performance between ChromaDB and FAISS
import time

test_query = "credit card billing problem"
k = 10

# Test FAISS performance
start_time = time.time()
faiss_results = vector_store.search_similar(test_query, k=k, use_faiss=True)
faiss_time = time.time() - start_time

# Test ChromaDB performance  
start_time = time.time()
chromadb_results = vector_store.search_similar(test_query, k=k, use_faiss=False)
chromadb_time = time.time() - start_time

print(f"Search Performance Comparison (k={k}):")
print(f"FAISS search time: {faiss_time:.4f} seconds")
print(f"ChromaDB search time: {chromadb_time:.4f} seconds")
print(f"FAISS speedup: {chromadb_time/faiss_time:.1f}x faster")

# Compare top results
print(f"\nTop 3 FAISS results:")
for i, result in enumerate(faiss_results[:3], 1):
    print(f"{i}. Score: {result['similarity_score']:.3f} | {result['product_category']}")

print(f"\nTop 3 ChromaDB results:")
for i, result in enumerate(chromadb_results[:3], 1):
    print(f"{i}. Score: {result['similarity_score']:.3f} | {result['product_category']}")

## Summary

**Task 2 Implementation Summary**:

1. **Stratified Sampling**: Successfully created a representative 12,000-complaint sample maintaining proportional distribution across all product categories.

2. **Text Chunking**: Implemented recursive character-based chunking with 500-character chunks and 50-character overlap, optimizing for semantic coherence while maintaining computational efficiency.

3. **Embedding Generation**: Used `sentence-transformers/all-MiniLM-L6-v2` model to generate 384-dimensional embeddings, chosen for its balance of performance, efficiency, and domain suitability.

4. **Vector Store Creation**: Built both ChromaDB and FAISS implementations with comprehensive metadata storage, enabling efficient semantic search and source traceability.

The pipeline successfully processes complaint narratives into a searchable vector format, ready for RAG implementation in Task 3.